#### Section C: Question 5:    (15 Marks)

Develop a Semantic segmentation model using Unet architecture on the given dataset.

Dataset contains the images and the corresponding masks. Find the dataset under the folder “Unet_Dataset”. Note that the masks are binary. Define the architecture accordingly.

Students can make use of pre-trained Unet segmentation model using the library

import segmentation_models as sm

Hints

1. Load all the images in one array of size 150x128x128x3
    Where 150 is total number of trained images
    128x128x3 is each image size
2. Load all the masks in one array of size 150x128x128x1
3. Scale both the above two arrays
4. Split the data into train and test
5. Define the pre-trained segmentation model
6. Compile with appropriate loss and metric and fit the data into it.

Run the model for minimum 5 epochs and present your result. The solution will be evaluated based on approach only as it take lot of epochs to produce good result. 


In [1]:
import os
import cv2
from PIL import Image
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt
import segmentation_models as sm
from sklearn.model_selection import train_test_split

2025-01-23 11:48:45.432900: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-23 11:48:46.692313: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


ModuleNotFoundError: No module named 'segmentation_models'

In [2]:
image_dir='Unet_Dataset/images/'

mask_dir='Unet_Dataset/MASKS_BW/'

In [4]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Function to load all images into a numpy array
def load_images_from_folder(folder, target_size):
    images = []
    for filename in sorted(os.listdir(folder)):  # Sorting ensures correct pairing
        img_path = os.path.join(folder, filename)
        img = load_img(img_path, target_size=target_size)
        img_array = img_to_array(img)
        images.append(img_array)
    return np.array(images)

In [9]:
# Parameters
IMG_HEIGHT = 128
IMG_WIDTH = 128
IMG_CHANNELS = 3
TOTAL_IMAGES = 150

# Step 1: Load all the images
images = load_images_from_folder(image_dir, target_size=(IMG_HEIGHT, IMG_WIDTH))
masks = load_images_from_folder(mask_dir, target_size=(IMG_HEIGHT, IMG_WIDTH))
masks = np.expand_dims(masks[..., 0], axis=-1)  # Keep masks binary (128x128x1)

len(images), len(masks)

(150, 150)

In [11]:
from sklearn.model_selection import train_test_split

# 2. Scale images and masks
images = images / 255.0  # Scale images to range [0, 1]
masks = masks / 255.0    # Scale masks to range [0, 1]

# 3. Split the data into train and test
X_train, X_test, y_train, y_test = train_test_split(images, masks, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((120, 128, 128, 3), (30, 128, 128, 3))

In [ ]:
# pip install if this doesn't work
# Required for newer versions of TensorFlow
os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

# 4. Define the pre-trained segmentation model (U-Net)
BACKBONE = 'resnet34'  # Using ResNet34 backbone
preprocess_input = sm.get_preprocessing(BACKBONE)

# Preprocess data
X_train = preprocess_input(X_train)
X_test = preprocess_input(X_test)

# Define model
model = sm.Unet(BACKBONE, encoder_weights='imagenet', input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), classes=1, activation='sigmoid')

# 5. Compile the model
model.compile(optimizer='adam',
              loss=sm.losses.bce_jaccard_loss,  # Binary crossentropy + Jaccard loss
              metrics=[sm.metrics.iou_score])  # Intersection over Union (IoU)

Segmentation Models: using `tf.keras` framework.


2025-01-23 11:56:57.340427: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-01-23 11:56:57.853494: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


85521592/85521592 [==============================] - 4s 0us/step


In [17]:
# 6. Train the model
# This willllll take time
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,  # Run for at least 5 epochs
    batch_size=8
)

# Evaluate the model
results = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Loss: {results[0]}")
print(f"Test IoU: {results[1]}")

# Save the model
# model.save("unet_segmentation_model.h5")

Epoch 1/5
15/15 [==============================] - 22s 919ms/step - loss: 1.3297 - iou_score: 6.8365e-04 - val_loss: 1.7470 - val_iou_score: 5.5862e-04
Epoch 2/5
15/15 [==============================] - 12s 800ms/step - loss: 1.1157 - iou_score: 6.8203e-04 - val_loss: 1.2160 - val_iou_score: 3.1352e-04
Epoch 3/5
15/15 [==============================] - 12s 794ms/step - loss: 1.0602 - iou_score: 6.6879e-04 - val_loss: 1.0355 - val_iou_score: 2.4466e-04
Epoch 4/5
15/15 [==============================] - 12s 788ms/step - loss: 1.0386 - iou_score: 6.4958e-04 - val_loss: 1.2027 - val_iou_score: 6.8998e-04
Epoch 5/5
1/1 [==============================] - 0s 337ms/step - loss: 1.0622 - iou_score: 5.3484e-04
Test Loss: 1.0621737241744995
Test IoU: 0.0005348432459868491


/home/vishrut/.pyenv/versions/scrape/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
